In [17]:
# Mengimpor pustaka yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBRegressor  # Pastikan Anda sudah menginstal xgboost
import warnings

# Menonaktifkan peringatan
warnings.filterwarnings('ignore')

# Memuat dataset diet dari file
try:
    # --- PENTING: Gunakan path lengkap Anda ---
    data = pd.read_csv("C:/Uner/Semester 5/Multivariat/Coolyeah/Tugas Kelompok 3/diet.csv")
except FileNotFoundError:
    print("Error: Pastikan file 'diet.csv' ada di direktori yang benar.")
    exit()

# --- DAFTAR VARIABEL TELAH DIPERBAIKI (Endogen diganti) ---
relevant_columns = [
    # Faktor 1: Makronutrien (5 Eksogen)
    'DR1TKCAL', 'DR1TPROT', 'DR1TCARB', 'DR1TTFAT', 'DR1TSUGR',
    
    # Faktor 2: Mineral (5 Eksogen)
    'DR1TCALC', 'DR1TIRON', 'DR1TSODI', 'DR1TMAGN', 'DR1TPHOS',
    
    # Faktor 3: Vitamin (5 Eksogen)
    'DR1TVC', 'DR1TVB6', 'DR1TFA', 'DR1TVD', 'DR1TVB12',
    
    # Faktor 4: Lemak & Lainnya (5 Eksogen)
    'DR1TCHOL', 'DR1TCAFF', 'DR1TSFAT', 'DR1TMFAT', 'DR1TPFAT',
    
    # Faktor 5: Perilaku_Diet (5 Endogen) - BARU
    'DBD100',      # Keep - Frekuensi tambah garam
    'DBQ095Z',     # Keep - Jenis garam
    'DR1STY',      # Baru - Tambah garam kemarin?
    'DR1DRSTZ',     # Baru - Jumlah makan vs biasa?
    'DR1BWATZ'     # Baru - Jumlah air kemasan?
]

print(f"Total kolom yang dipilih: {len(relevant_columns)}")

# Memeriksa apakah semua kolom baru ini ada
missing_cols = [col for col in relevant_columns if col not in data.columns]
if missing_cols:
    print(f"\n--- PERINGATAN ---")
    print(f"Error: Kolom berikut TIDAK DITEMUKAN di 'diet.csv': {missing_cols}")
    # Melanjutkan hanya dengan kolom yang ada
    relevant_columns = [col for col in relevant_columns if col in data.columns]
    print(f"Melanjutkan dengan {len(relevant_columns)} kolom yang ditemukan.")

if not relevant_columns:
    print("Tidak ada kolom relevan yang ditemukan. Menghentikan eksekusi.")
    exit()
elif len(relevant_columns) != 25:
     print(f"PERINGATAN: Hanya {len(relevant_columns)} dari 25 kolom yang ditemukan!")


# Membuat subset data dengan hanya kolom relevan
subset_data = data[relevant_columns].copy()

# Mengonversi kolom menjadi numerik
for col in subset_data.columns:
    subset_data[col] = pd.to_numeric(subset_data[col], errors='coerce')

print(f"\n--- Info Data Awal (Memproses {subset_data.shape[1]} kolom) ---")
subset_data.info()

# Inisialisasi IterativeImputer dengan XGBRegressor
imputer = IterativeImputer(
    estimator=XGBRegressor(objective='reg:squarederror'),
    max_iter=10,
    random_state=0,
    verbose=2
)

print("\nMemulai imputasi dengan IterativeImputer (Estimator: XGBRegressor)...")
# Terapkan imputer
imputed_array = imputer.fit_transform(subset_data)

# Ubah kembali ke DataFrame
imputed_data = pd.DataFrame(imputed_array, columns=relevant_columns)
print("Imputasi selesai.")

print("\n--- Info Data Setelah Imputasi ---")
imputed_data.info()

# (Opsional: Tambahkan kembali visualisasi jika diinginkan)
# def visualize_imputation(...): ...
# visualize_imputation(subset_data, imputed_data, 'DBD100') # Contoh var endogen baru

# Menyimpan subset data yang telah disiapkan ke file baru
output_filename = 'diet_final.csv' # Nama file baru
imputed_data.to_csv(output_filename, index=False)

print(f"\nDataset subset dengan variabel endogen baru disimpan sebagai '{output_filename}'")

Total kolom yang dipilih: 25

--- Info Data Awal (Memproses 25 kolom) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9813 entries, 0 to 9812
Data columns (total 25 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   DR1TKCAL  8531 non-null   float64
 1   DR1TPROT  8531 non-null   float64
 2   DR1TCARB  8531 non-null   float64
 3   DR1TTFAT  8531 non-null   float64
 4   DR1TSUGR  8531 non-null   float64
 5   DR1TCALC  8531 non-null   float64
 6   DR1TIRON  8531 non-null   float64
 7   DR1TSODI  8531 non-null   float64
 8   DR1TMAGN  8531 non-null   float64
 9   DR1TPHOS  8531 non-null   float64
 10  DR1TVC    8531 non-null   float64
 11  DR1TVB6   8531 non-null   float64
 12  DR1TFA    8531 non-null   float64
 13  DR1TVD    8531 non-null   float64
 14  DR1TVB12  8531 non-null   float64
 15  DR1TCHOL  8531 non-null   float64
 16  DR1TCAFF  8531 non-null   float64
 17  DR1TSFAT  8531 non-null   float64
 18  DR1TMFAT  8531 non-null   float64


PermissionError: [Errno 13] Permission denied: 'diet_final.csv'